# **Justice for All? An Empirical Analysis of Fairness and Consistency in U.S. Federal Sentencing**

##### **A.J. Beiza, A.J. Clark, Lucas Fierro Ruiz, Austin Hayes, Tyler Komito, Smirthi Murali, & Dylan Wilson**

#### This notebook contains the source code used to generate the findings presented in our DTSC 4301/4302 capstone project for the School of Data Science at the University of North Carolina at Charlotte.





## Import Required Libraries

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import scipy.stats as stats
import plotly.express as px

import os
import requests
from pathlib import Path
from typing import List, Tuple, Dict, Any, Union
from collections import defaultdict
from itertools import combinations, product
import sys
sys.path.append(os.path.join(str(Path.cwd()), "../"))  # allows importing from parent directory

from scripts.data_download import download_files_from_fld
import scripts.census as census

## Downloading Data

##### Run the following cell to import all the data necessary to run this Jupyter Notebook.

In [6]:
folder_id = "1P2FRAkPrqL2nn2MNMyd4ilWbXNS_kkKD"  # Folder ID on Google Drive
data_folder = os.path.join(str(Path.cwd()), "../data")
download_files_from_fld(folder_id, data_folder)

Retrieving folder contents


Processing file 1JATvvOEw2llpX52TuAeza9C76YnBPiIj county_to_jud_district_xref.csv
Processing file 1r9L2VUSg_j9t2CsSYOnyi3HzYXyyd64U election_results_2000-2024_MIT.csv
Processing file 1xvJk8290a5fluZaUpqoFbnZz9bc6S2t4 opafy14_24_combined_filtered.csv
Processing file 1l7nAYrXHOMcLnyAf5FpcfmKL1A-9f1_U us_district_cts_bounds.geojson


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1JATvvOEw2llpX52TuAeza9C76YnBPiIj
To: c:\Users\austy\USSC-Fairness-Analysis-DTSC-430X-Capstone-\data\county_to_jud_district_xref.csv
100%|██████████| 320k/320k [00:00<00:00, 6.71MB/s]
Downloading...
From: https://drive.google.com/uc?id=1r9L2VUSg_j9t2CsSYOnyi3HzYXyyd64U
To: c:\Users\austy\USSC-Fairness-Analysis-DTSC-430X-Capstone-\data\election_results_2000-2024_MIT.csv
100%|██████████| 10.2M/10.2M [00:01<00:00, 6.49MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1xvJk8290a5fluZaUpqoFbnZz9bc6S2t4
From (redirected): https://drive.google.com/uc?id=1xvJk8290a5fluZaUpqoFbnZz9bc6S2t4&confirm=t&uuid=9eb54b70-4058-4734-b06b-02c7ef26422a
To: c:\Users\austy\USSC-Fairness-Analysis-DTSC-430X-Capstone-\data\opafy14_24_combined_filtered.csv
100%|██████████| 291M/291M [00:33<00:00, 8.67MB/s] 
Downloading...
From: https://driv

In [7]:
# example usage of census module to get county data for all years from 2018 to 2021
census_df = census.get_county_data_all_yrs(
    start_yr=2022,
    end_yr=2024,
    var_codes=["DP05_0001E"], 
    var_names=["Total Population"],
    api_key=None,
    # filename="county_population_2018_2021.csv"
)
census_df["Total Population"] = census_df["Total Population"].astype(int)
county_to_jud_xref = pd.read_csv(os.path.join(data_folder, "county_to_jud_district_xref.csv"))
census_df = (
    census_df.merge(county_to_jud_xref, how='left', on="geoid")
    [["CEN_YR", "district", "Total Population"]]
    .groupby(["CEN_YR", "district"])["Total Population"]
    .sum()
    .reset_index()
)
census_df.head()

,CEN_YR,district,Total Population
0,2022,Alaska,725207
1,2022,Arizona,7172282
2,2022,Central,21461552
3,2022,Colorado,5770790
4,2022,Delaware,993635


In [8]:
sent_df = pd.read_csv(os.path.join(data_folder, "opafy14_24_combined_filtered.csv"), encoding='latin1')
sent_df.head()

C:\Users\austy\AppData\Local\Temp\ipykernel_7548\171328483.py:1: DtypeWarning: Columns (0,112,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279) have mixed types. Specify dtype option on import or set low_memory=False.
  sent_df = pd.read_csv(os.path.join(data_folder, "opafy14_24_combined_filtered.csv"), encoding='latin1')


,POOFFICE,ZONE,COSTSUP,FINE,FINEMIN,FINEMAX,SENTTOT,SENSPLT0,TIMSERVC,TOTREST,...,RETEXT23,RETEXT24,RETEXT25,MNTHDEPT,ENCRYPT1,ENCRYPT2,FY,OFFTYPSB,SENTMON,SENTYR
0,6,D,0.000000e+00,1000.0,NaN,NaN,15.0,15.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,2014,NaN,10.0,2013.0
1,6,D,0.000000e+00,1000.0,NaN,NaN,84.0,84.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,2014,NaN,10.0,2013.0
2,2,C,1.000000e+10,0.0,NaN,NaN,7.0,7.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,2014,NaN,10.0,2013.0
3,2,D,0.000000e+00,0.0,NaN,NaN,120.0,120.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,2014,NaN,10.0,2013.0
4,5,D,0.000000e+00,0.0,NaN,NaN,30.0,30.0,0.0,0.0,...,NaN,NaN,NaN,NaN,0.0,0.0,2014,NaN,10.0,2013.0


In [9]:
# Remove rows where the USSC did not recieve sentencing information documents from the district courts
sent_df = sent_df[
    (sent_df["DSSOR"] == 1) 
    & (sent_df["DSPSR"] == 1)
    & (sent_df["DSJANDC"] == 1)
]
print("Shape after filtering:", sent_df.shape)

Shape after filtering: (673931, 288)


In [10]:
min_cols = [col for col in sent_df.columns if "MIN" in col]

In [11]:

# this suggests we can drop the MIN columns, there are also no rows where SEXMIN > GLMIN
sent_df[sent_df["SEXMIN"]!=0][["SEXMIN", "GLMIN"]].head(20)

,SEXMIN,GLMIN
3,120,120.0
226,120,120.0
227,120,188.0
387,180,1800.0
408,120,120.0
783,180,9996.0
824,120,324.0
944,120,135.0
1097,120,120.0
3036,120,188.0


In [13]:
sent_df['POOFFICE'].unique()

array(['6', '2', '5', '1', '7', '3', '8', '4', 'E', 'G', '9', '0', 'C',
       'A', 'D', 2, 1, 4, 6, 3, 8, 5, 0, 7, 9, 'B', nan], dtype=object)